In [38]:
import jax
import jax.numpy as jnp 
from jax import random

In [40]:
device = "gpu" if jax.default_backend() == "gpu" else "cpu"

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [28]:
import os
import urllib.request

In [32]:
# data

In [33]:
block_size = 256
batch_size = 64
embeddings_dims = 384
attn_dropout = 0.1
no_of_heads = 8
dropout = 0.1
epochs = 100
max_lr = 3e-4
no_of_decoder_layers = 8 
attn_dropout = 0.1
weight_decay_optim = 0.01

In [ ]:
def init_model(key, vocab_size):
    k1, k2, k3 = random.split(key, 3)

    params = {
        "embeddings": init_embeddings(k1, vocab_size, embeddings_dims),
        "ln": init_layernorm(embeddings_dims),
        "mlp": init_MLPBlock_params(k2)
    }

    return params

In [ ]:
def forward(params, x, key, training=True):
    key, k1 = random.split(key)

    x = embedding_forward(params["embeddings"], x)
    x = layernorm_forward(params["ln"], x)
    x = MLPBlock(params["mlp"], x, k1, training=training)

    return x

In [ ]:
def init_embeddings(key, vocab_size, embedding_dim):
    return jax.random.normal(key, (vocab_size, embedding_dim)) * 0.02

def embedding_forward(params, x):
    # x: (batch, seq_len) or any int indices
    return params[x]

In [ ]:

def init_layernorm(embedding_dim):
    # gamma (scale) and beta (shift)
    params = {
        "gamma": jnp.ones((embedding_dim,)),
        "beta": jnp.zeros((embedding_dim,))
    }
    return params


def layernorm_forward(params, x, eps=1e-5):
    mean = jnp.mean(x, axis=-1, keepdims=True)
    var = jnp.var(x, axis=-1, keepdims=True)
    x_hat = (x - mean) / jnp.sqrt(var + eps)
    return params["gamma"] * x_hat + params["beta"]

In [ ]:
def init_MLPBlock_params(key, embeddings_size=embeddings_dims):
    hidden_size = 4 * embeddings_dims
    k1, k2 = random.split(key)

    params = {
        "w1": random.normal(k1, (embeddings_size, hidden_size)) * 0.02,
        "b1": jnp.zeros((hidden_size,)),
        "w2": random.normal(k2, (hidden_size, embeddings_size)) * 0.02,
        "b2": jnp.zeros((embeddings_size,))
    }
    return params


def MLPBlock(params, x, key, dropout=dropout, embeddings_size=embeddings_dims, training=True):
    x = x @ params["w1"] + params["b1"] #l1
    x = jax.nn.gelu(x) #gelu
    x = x @ params["w2"] + params["b2"] #l2
    if training: #dropout
        keep_prob = 1.0 - dropout
        mask = random.bernoulli(key, keep_prob, x.shape)
        x = jnp.where(mask, x / keep_prob, 0)

    return x

In [41]:
def init_attention_head(key, embeddings_dims, head_size):
    k1, k2, k3 = random.split(key, 3)
    scale = 0.02  
    
    params = {
        "query": random.normal(k1, (embeddings_dims, head_size)) * scale,
        "keys": random.normal(k2, (embeddings_dims, head_size)) * scale,
        "values": random.normal(k3, (embeddings_dims, head_size)) * scale,
    }
    return params

def attention_head_forward(params, x, key, attn_dropout=0.1, training=True):
    batch, block_size, embd_dims = x.shape
    q = x @ params["query"]  # (batch, block_size, head_size)
    k = x @ params["keys"]    # (batch, block_size, head_size)
    v = x @ params["values"]  # (batch, block_size, head_size)
    scale = k.shape[-1] ** -0.5
    weights = q @ jnp.transpose(k, (0, 2, 1)) * scale  # (batch, block_size, block_size)
    mask = jnp.tril(jnp.ones((block_size, block_size)))
    mask = mask.at[:, :].set(mask == 0)  # Boolean mask where True = positions to mask
    masked_weights = jnp.where(mask, -jnp.inf, weights)
    weights_normalized = jax.nn.softmax(masked_weights, axis=-1)
    if training and attn_dropout > 0:  #dropout
        keep_prob = 1.0 - attn_dropout
        dropout_key, _ = random.split(key)
        mask_dropout = random.bernoulli(dropout_key, keep_prob, weights_normalized.shape)
        weights_normalized = jnp.where(mask_dropout, weights_normalized / keep_prob, 0)

    out = weights_normalized @ v  # (batch, block_size, head_size)
    return out